# 作业4 - 丁平尖
**日期**: 2026年6月22日

## 1 注意事项
1. 编程题需要打印相应的输出
2. 将 ipynb 文件提交至 github 上，命名为：HW04-学号-姓名.ipynb

## 2 序列模型

### 2.1 理论计算题

给定一个字符序列"ababc"，假设采用一阶马尔可夫模型（即 $p(x_t|x_{t-1})$）使用拉普拉斯平滑（加1平滑）估计以下条件概率：

1. $p(a|b)$
2. $p(c|b)$

（词汇表为 {a, b, c}，计算时考虑所有可能转移，包括未出现的情况。）

**解答：**

转移计数（不加平滑）：
- 从 a → b: 出现2次（位置1→2, 3→4）
- 从 b → a: 出现1次（位置2→3）
- 从 b → c: 出现1次（位置4→5）
- 其他转移: 0次

平滑后条件概率公式：
$$p(x_t | x_{t-1}) = \frac{\text{count}(x_{t-1}, x_t) + 1}{\text{count}(x_{t-1}) + |V|}$$

其中 |V| = 3。

count(a) = 2, count(b) = 2, count(c) = 1

1. $p(a|b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$

2. $p(c|b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$

### 2.2 编程题

编写一个函数preprocess_text(text,n)，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）
2. 按空格分词
3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
4. 用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签（用于自回归语言模型）

返回词汇表字典和（特征列表，标签列表）。

In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本并生成自回归特征序列
    
    参数:
        text: 输入字符串
        n: 滑动窗口大小（特征序列长度）
    
    返回:
        vocab: 字典，词到ID的映射
        features: 特征列表，每个特征是长度为n的词列表
        labels: 标签列表，每个标签是下一个词的ID
    """
    # 1. 转小写，去除非字母和空格的字符
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按频率排序，从0开始分配ID）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        feature = words[i:i+n]
        label_word = words[i+n]
        features.append(feature)
        labels.append(vocab[label_word])
    
    return vocab, features, labels

# 测试
text = "The time machine"
vocab, features, labels = preprocess_text(text, 2)

print("词汇表:", vocab)
print("特征:", features)
print("标签ID:", labels)
print("标签词:", [list(vocab.keys())[l] for l in labels])

## 3 循环神经网络

### 3.1 理论计算题

考虑一个线性 RNN（无偏置），定义为 $h_t = W_{hh}h_{t-1} + W_{hx}x_t$，输出 $o_t = W_{oh}h_t$。假设损失函数为平方损失 $L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

**解答：**

定义 $\delta_t = \frac{\partial L}{\partial o_t} = o_t - y_t$

由于 $o_t = W_{oh}h_t$，所以 $\frac{\partial L}{\partial h_t} = W_{oh}^T \delta_t$

并且 $h_t$ 不仅影响 $o_t$，还通过 $h_{t+1}$ 影响后续所有输出：

$$\frac{\partial L}{\partial h_t} = W_{oh}^T \delta_t + W_{hh}^T \frac{\partial L}{\partial h_{t+1}}$$

展开到所有时间步：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} h_{t-1}^T$$

其中 $\frac{\partial L}{\partial h_t} = \sum_{k=t}^T (W_{hh}^T)^{k-t} W_{oh}^T \delta_k$

**梯度消失/爆炸条件：**
- **梯度爆炸**：如果 $W_{hh}$ 的谱半径 > 1，且 $T-t$ 较大，梯度会指数增长
- **梯度消失**：如果 $W_{hh}$ 的谱半径 < 1，且 $T-t$ 较大，梯度会指数衰减到0

梯度爆炸或消失取决于 $W_{hh}$ 的特征值的模是否大于1或小于1。

### 3.2 编程题

实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。

In [ ]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        W_xh: 输入到隐藏权重，形状 (input_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 前向计算
    h_t = np.tanh(np.dot(x_t, W_xh) + np.dot(h_prev, W_hh) + b_h)
    
    # 缓存
    cache = (x_t, h_prev, h_t, W_hh, W_xh, b_h)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播（单步）
    
    参数:
        dh_next: 上游梯度，即损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    返回:
        dx_t: 损失对x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        dW_xh: 损失对W_xh的梯度，形状 (input_size, hidden_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, h_t, W_hh, W_xh, b_h = cache
    
    # tanh的导数: dtanh = 1 - tanh^2
    dtanh = dh_next * (1 - h_t**2)
    
    # 对W_hh的梯度
    dW_hh = np.dot(h_prev.T, dtanh)
    
    # 对W_xh的梯度
    dW_xh = np.dot(x_t.T, dtanh)
    
    # 对b_h的梯度
    db_h = np.sum(dtanh, axis=0)
    
    # 对x_t的梯度
    dx_t = np.dot(dtanh, W_xh.T)
    
    # 对h_prev的梯度
    dh_prev = np.dot(dtanh, W_hh.T)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h

# 测试
batch_size = 2
input_size = 4
hidden_size = 3

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_xh = np.random.randn(input_size, hidden_size)
b_h = np.random.randn(hidden_size)

# 前向
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
print("h_t shape:", h_t.shape)

# 反向
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(dh_next, cache)

print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hh shape:", dW_hh.shape)
print("dW_xh shape:", dW_xh.shape)
print("db_h shape:", db_h.shape)

## 4 高级循环神经网络

### 4.1 理论计算题

假设一个深度双向 RNN，有 L 层，每层隐藏单元数为 H，输入维度为 D，输出维度为 O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

**解答：**

第1层参数：
- 前向: D×H + H² + 2H
- 后向: D×H + H² + 2H
- 合计: 2D×H + 2H² + 4H

第2层到第L层（共L-1层）：
- 每层输入维度为2H
- 每层参数 = 2 × (2H×H + H² + 2H) = 6H² + 4H
- (L-1)层合计: (L-1)×(6H² + 4H)

**总参数：**

$$\text{Total} = (2DH + 2H^2 + 4H) + (L-1)(6H^2 + 4H)$$
$$= 2DH + (6L - 4)H^2 + 4LH$$

### 4.2 编程题

实现一个双向 RNN 编码器，接收序列 X（形状（seq_len，batch，input_dim）），使用 torch.nn.RNN 或手动实现。

In [ ]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        """
        参数:
            X: 形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 形状 (seq_len, batch, 2*hidden_dim)
            final_state: 形状 (batch, 2*hidden_dim)
        """
        outputs, h_n = self.rnn(X)
        
        forward_last = h_n[-2, :, :]
        backward_last = h_n[-1, :, :]
        final_state = torch.cat([forward_last, backward_last], dim=1)
        
        return outputs, final_state

# 手动实现版本
class BidirectionalRNNEncoderManual(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn_forward = nn.RNNCell(input_dim, hidden_dim)
        self.rnn_backward = nn.RNNCell(input_dim, hidden_dim)
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        seq_len, batch, _ = X.shape
        
        h_forward = torch.zeros(batch, self.hidden_dim, device=X.device)
        h_backward = torch.zeros(batch, self.hidden_dim, device=X.device)
        
        forward_outputs = []
        backward_outputs = []
        
        for t in range(seq_len):
            h_forward = self.rnn_forward(X[t], h_forward)
            forward_outputs.append(h_forward.unsqueeze(0))
        
        for t in range(seq_len-1, -1, -1):
            h_backward = self.rnn_backward(X[t], h_backward)
            backward_outputs.insert(0, h_backward.unsqueeze(0))
        
        forward_outputs = torch.cat(forward_outputs, dim=0)
        backward_outputs = torch.cat(backward_outputs, dim=0)
        outputs = torch.cat([forward_outputs, backward_outputs], dim=2)
        final_state = torch.cat([forward_outputs[-1], backward_outputs[-1]], dim=1)
        
        return outputs, final_state

# 测试
seq_len, batch, input_dim = 5, 3, 10
hidden_dim = 4
X = torch.randn(seq_len, batch, input_dim)

# PyTorch内置版本
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
outputs, final_state = encoder(X)
print("PyTorch内置版本:")
print("outputs shape:", outputs.shape)
print("final_state shape:", final_state.shape)

# 手动实现版本
encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
outputs_man, final_state_man = encoder_manual(X)
print("\n手动实现版本:")
print("outputs shape:", outputs_man.shape)
print("final_state shape:", final_state_man.shape)

## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $\mathbf{v}_c, \mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$，写出完整的目标函数。

**解答：**

**目标函数**（最大化）：

$$\mathcal{L} = \log \sigma(\mathbf{u}_o^T \mathbf{v}_c) + \sum_{k=1}^K \mathbb{E}_{n_k \sim P_n} \left[ \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) \right]$$

等价地，最小化负对数似然（损失函数）：

$$\mathcal{J} = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$$

其中 $\sigma(x) = 1/(1+e^{-x})$ 是sigmoid函数。

**负样本采样**：

从噪声分布 $P_n(w)$ 中采样 K 个词作为负样本。常用的噪声分布是词频的3/4次方：

$$P_n(w) = \frac{\text{count}(w)^{0.75}}{\sum_{w'} \text{count}(w')^{0.75}}$$

这样可以降低高频词的采样概率，提高低频词的采样概率。

### 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整 softmax）。

In [ ]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_idx, W_in, W_out):
    """
    CBOW模型前向传播和损失计算（完整softmax）
    
    参数:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_idx: 目标词索引，形状 (batch_size,)
        W_in: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失，标量
    """
    # 1. 获取上下文的嵌入向量
    context_embeds = W_in[context_indices]
    
    # 2. 平均得到隐藏层
    h = torch.mean(context_embeds, dim=1)
    
    # 3. 计算输出得分
    scores = torch.matmul(h, W_out)
    
    # 4. 计算交叉熵损失
    loss = F.cross_entropy(scores, target_idx)
    
    return loss

# 测试
V, d = 10, 5
batch_size, context_size = 4, 3

W_in = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)

context_indices = torch.randint(0, V, (batch_size, context_size))
target_idx = torch.randint(0, V, (batch_size,))

loss = cbow_forward(context_indices, target_idx, W_in, W_out)
print("CBOW损失值:", loss.item())

loss.backward()
print("W_in梯度形状:", W_in.grad.shape)
print("W_out梯度形状:", W_out.grad.shape)

## 6 注意力机制

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 score = $Q K^T / \sqrt{d_k}$（$d_k = 4$）。可以只列出数值计算过程（用符号或具体数值）。

**解答：**

**步骤1：计算得分矩阵 S**

$$S = \frac{Q K^T}{\sqrt{d_k}} = \frac{Q K^T}{2}$$

其中 $Q \in \mathbb{R}^{2 \times 4}$，$K^T \in \mathbb{R}^{4 \times 3}$，所以 $S \in \mathbb{R}^{2 \times 3}$

$$S_{ij} = \frac{1}{2} \sum_{m=1}^4 Q_{i,m} K_{j,m}$$

**步骤2：对每行应用Softmax**

$$A_{ij} = \frac{\exp(S_{ij})}{\sum_{l=1}^3 \exp(S_{il})}$$

$A \in \mathbb{R}^{2 \times 3}$

**步骤3：加权求和得到输出**

$$\text{Output} = A \cdot V \in \mathbb{R}^{2 \times 5}$$

$$\text{Output}_{i, n} = \sum_{j=1}^3 A_{ij} V_{j, n}$$

### 6.2 编程题

实现多头注意力(Multi-Head Attention)的前向传播，假设 num_heads=2, d_model=4。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        """
        参数:
            X: 形状 (seq_len, batch, d_model)
        
        返回:
            output: 形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 分割成多头
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)
        
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 3. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        # 4. 合并多头
        attn_output = attn_output.permute(2, 0, 1, 3)
        attn_output = attn_output.contiguous().view(seq_len, batch, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(attn_output)
        
        return output

# 测试
d_model = 4
num_heads = 2
seq_len, batch = 6, 3

X = torch.randn(seq_len, batch, d_model)

mha = MultiHeadAttention(d_model, num_heads)
output = mha(X)

print("输入形状:", X.shape)
print("输出形状:", output.shape)
print("输入和输出形状相同:", X.shape == output.shape)

## 总结

所有题目已完成！